## Домашнее задание 4: Рекуррентные нейронные сети

В этом задании вам предстоит самостоятельно реализовать модель GRU для решения задачи классификации с пересекающимися классами (multi-label classification). Это вид классификации, в которой каждый объект может относиться одновременно к нескольким классам. Данная задача может возникнуть при классификации фильмов по жанрам, научных или новостных статей по темам, музыкальных композиций по инструментам и так далее.

В нашем случае мы будем работать с датасетом биотехнических новостей и классифицировать их по темам. Этот датасет уже предобработан: текст приведен к нижнему регистру, удалена пунктуация, все слова разделены проблелом.

In [1]:
import pandas as pd
import numpy as np

In [2]:
dataset = pd.read_csv('/kaggle/input/biotech-news/biotech_news.tsv', sep='\t')
dataset.head()

,text,labels
0,drive your plow over the bones of the dead by ...,other
1,in the recently tabled national budget denel h...,other
2,shares take a break its good for you picture g...,other
3,reso is currently hiring for two positions pro...,other
4,charter buyer club what is the charter buyer c...,other


## Предобработка лейблов

Как вы модете заметить, лейблы записаны в виде строк, разделенных запятыми. Для работы с ними нам нужно преобразовать их в числа. Так как каждый объект может принадлежать нескольким классам, закодируем лейблы в виде векторов из 0 и 1, где 1 означает, что объект принадлежит соответствующему классу, а 0 – не принадлежит. Имея такую кодировку, мы сможем обучить модель, решая задачу бинарной классификации для каждого класса.

In [3]:
all_labels = set()
for labels in dataset['labels']:
    all_labels |= set(labels.split(', '))

all_labels = sorted(all_labels)

all_labels, len(all_labels)

(['alliance & partnership',
  'article publication',
  'clinical trial sponsorship',
  'closing',
  'company description',
  'department establishment',
  'event organization',
  'executive appointment',
  'executive statement',
  'expanding geography',
  'expanding industry',
  'foundation',
  'funding round',
  'hiring',
  'investment in public company',
  'ipo exit',
  'm&a',
  'new initiatives & programs',
  'new initiatives or programs',
  'other',
  'participation in an event',
  'partnerships & alliances',
  'patent publication',
  'product launching & presentation',
  'product updates',
  'regulatory approval',
  'service & product providing',
  'subsidiary establishment',
  'support & philanthropy'],
 29)

In [4]:
name2id = {name: i for i, name in enumerate(all_labels)}
id2name = {i: name for name, i in name2id.items()}

In [5]:
def binarize_labels(labels):
    numeric_label = np.zeros(len(name2id), dtype=int)
    for name in labels.split(', '):
        numeric_label[name2id[name]] = 1

    return numeric_label

In [6]:
numeric_labels = dataset['labels'].apply(binarize_labels)
numeric_labels[-4:]

3035    [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
3036    [0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, ...
3037    [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...
3038    [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, ...
Name: labels, dtype: object

## Предобработка данных

В этом задании мы будем обучать рекуррентные нейронные сети. Как мы знаем, они работают хуже для длинных текстов. Поэтому, удалим из текстов стоп слова, слишком редкие, а также слишком частые слова. Все эти слова не должны влиять на класс текста.

In [7]:
texts = dataset['text'].apply(lambda x: x.split())

Сразу разделим выборку на обучающую и тестовую, чтобы считать статистики только по обучающей.

In [8]:
from sklearn.model_selection import train_test_split

texts_train, texts_test, y_train, y_test = train_test_split(texts, numeric_labels, test_size=0.2, random_state=0)

__Задание 1.__ Напишите функцию `process_datasets`, которая принимает на вход обучающую выборку `texts_train`, тестовую выборку `texts_test`, а также параметры `min_wf` и `max_wf`. Функция считает встречаемость каждого слова по тренировочной выборке и удаляет из обоих выборок все __стоп слова__, а также слова, которые встречаются меньше `min_wf` раз и больше `max_wf`. Функция возвращает обработанные выборки в том же порядке.

In [9]:
# import nltk
# nltk.download('stopwords')

In [10]:
from nltk.corpus import stopwords

stop_words = stopwords.words('english')

def process_datasets(texts_train, texts_test, min_wf, max_wf):
    # ваш код здесь
    from collections import Counter
    cnt = Counter()
    for seq in texts_train:
        cnt.update(seq)
    items = list(cnt.items())
    for word, count in items:
        if count < min_wf or count > max_wf:
            del cnt[word]

    processed_texts_train = []
    for seq in texts_train:
        new_seq = [w for w in seq if w in cnt and w not in stop_words]
        processed_texts_train.append(new_seq)

    processed_texts_test = []
    for seq in texts_test:
        new_seq = [w for w in seq if w in cnt and w not in stop_words]
        processed_texts_test.append(new_seq)
        
    return processed_texts_train, processed_texts_test

Удалим все слова, которые встречаются меньше 4 раз в выборке, а также встречаемость которых больше, чем 95% от размера датасета.

In [11]:
texts_train, texts_test = process_datasets(
    texts_train, texts_test, min_wf=4, max_wf=int(0.95 * len(texts_train))
)

Создадим словарь, который будем использовать для конвертации слов в индексы.

In [12]:
from gensim.corpora.dictionary import Dictionary

dictionary = Dictionary(texts_train)
dictionary.add_documents([['PAD', 'UNK']])
len(dictionary)

16480

Видим, что после обработки у нас осталось около 16 тысяч уникальных слов. Это отличное значение, поэтому больше ничего не будем делать с текстами.

In [13]:
import torch
from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    """
    Эта функция вызывается при формировании батча в DataLoader.
    Она токенизирует текст и добавляет паддинги.
    """
    texts, labels = zip(*batch)
    pad_token_id = dictionary.token2id['PAD']
    unk_token_id = dictionary.token2id['UNK']
    input_ids = [torch.tensor(dictionary.doc2idx(text, unknown_word_index=unk_token_id)) for text in texts]
    return (
        pad_sequence(input_ids, padding_value=pad_token_id, batch_first=True).long(),
        torch.tensor(labels).float()
    )

Обернем обе выборки в DataLoader и перейдем к обучению рекуррентных сетей.

In [14]:
train_dataset = list(zip(texts_train, y_train))
test_dataset = list(zip(texts_test, y_test))

In [15]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, collate_fn=collate_fn, shuffle=True, batch_size=64)
test_loader = DataLoader(test_dataset, collate_fn=collate_fn, shuffle=False, batch_size=64)

## Обучение моделей

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

### RNN

Напомним, что RNN – это простейшая рекуррентрая нейронная сеть и ее блок RNN выглядит таким образом

<img src="https://i.ibb.co/S5gfLzR/rnn.png" alt="drawing" width="400"/>

Его скрытое состояние обновляется по формуле
$
h_t = \sigma(W x_{t-1} + U h_{t-1} + b_h).
$   
А предсказание считается с помощью применения линейного слоя к последнему токену
$
o_T = O h_T + b_o.
$

Ниже представлена реализация RNN, взятая из практической части урока. Вы можете ее модифицировать, например, для ускорения, по своему усмотрению.

In [17]:
from torch import nn


class RNN(nn.Module):
    def __init__(self, vocab_size, n_classes, pad_token_id, input_size=256, hidden_size=256):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, input_size)

        self.hidden_size = hidden_size
        self.hidden_fc = nn.Linear(input_size + hidden_size, hidden_size)
        self.out_fc = nn.Linear(hidden_size, n_classes)

        self.pad_token_id = pad_token_id

    def forward(self, input_ids, init_h=None):
        bs, seq_len = input_ids.shape

        x = self.embedding(input_ids)

        if init_h is None:
            # инициализируем скрытое состояние нулями
            h_t = torch.zeros(bs, self.hidden_size, device=x.device)
        else:
            h_t = init_h

        hidden_states = []
        for t in range(seq_len):
            # обновляем скрытое состояние RNN и сохраняем его в массив
            x_t = x[:, t, :]
            cat_state = torch.cat((x_t, h_t), dim=1)
            h_t = torch.tanh(self.hidden_fc(cat_state))

            hidden_states.append(h_t.unsqueeze(1))

        # применяем линейный слой ко всем скрытым состояниям, чтобы получить логиты
        hidden_states = torch.cat(hidden_states, dim=1)

        sequence_lengths = (input_ids == self.pad_token_id).int().argmax(-1) - 1
        sequence_lengths = sequence_lengths % input_ids.shape[-1]

        last_hidden_states = hidden_states[torch.arange(len(input_ids)), sequence_lengths]
        logits = self.out_fc(last_hidden_states)

        return torch.sigmoid(logits)

Перед тем, как приступить к обучению, нам нужно выбрать метрику оценки качества. Так как в задаче классификации с пересекающимися классами классы часто несбалансированы, чаще всего в качестве метрики берется [F1 score](https://en.wikipedia.org/wiki/F-score).

__Задание 2.__ Напишите функцию `compute_f1`, которая принимает истинные метки и предсказанные и считает среднее значение F1 по всем классам.

$$
F1_{total} = \frac{1}{K} \sum_{k=1}^K F1(Y_k, \hat{Y}_k),
$$
где $Y_k$ – истинные значения для класса k, а $\hat{Y}_k$ – предсказания.

In [18]:
from sklearn.metrics import f1_score

def compute_f1(y_true, y_pred):
    # ваш код здесь
    K = len(y_true[0,:])
    f1_score_K = []
    for k in range(K):
        f1s = f1_score(y_true[:,k].cpu(), y_pred[:,k].cpu())
        f1_score_K.append(f1s)

    return sum(f1_score_K)/K

Осталось написать циклы обучения и валидации.

__Здадание 3.__ Допишите функцию `train` для обучения модели. Она принимает 5 параметров:
* `model`
* `dataloader`
* `optimizer`
* `device` – устройство используемое для обучения: `cpu/cuda`.
* `logging` – бинарная переменная. Если `logging = True`, то функция логирует процесс обучения. В противном случае логгирование не производится.

Функция обучает модель в течение одной эпохи на полученном датасете. Ошибка модели считается как средняя ошибка бинарной классификации для каждого класса.

In [19]:
def train(model, dataloader, optimizer, device='cpu', logging=False):
    """
    Обучает модель (model) на всем наборе данных (dataloader).
    """
    # ваш код здесь
    # не забываем переводить в train режим
    model.train()
    criterion = nn.BCELoss()

    for input_ids, labels in dataloader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)
        
        preds = model(input_ids)

        # переставляем размерности логитов, так как в CrossEntropyLoss
        # предсказания классов должны быть второй размерностью
        loss = criterion(preds, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        f1 = compute_f1((preds > 0.5).int().cpu(), labels)

        if logging:
            # логируем значения ошибки и f1
            print({
                "train_loss": loss.item(),
                "train_f1": f1
            })

__Здадание 4.__ Допишите функцию `evaluate` для тестирования модели. Она принимает 3 параметра: `model`, `dataloader` и `device`. Функция совершает предсказания для всех объектов в `dataloader`, считает по ним значение F1 и возвращает его. Не забывайте, что если считать F1 отдельно для каждого батча, а затем усреднять, то результат будет неверным.

Если вы хотите изменить поведение функции и, например, возвращать значение лосса или логировать F1 внутри, то вы можете это сделать после прохождения проверки.

In [20]:
@torch.inference_mode()
def evaluate(model, dataloader, device='cpu'):
    """
    Тестирует модель (model) на всем наборе данных (dataloader).
    """
    # ваш код здесь
    # не забываем переводить в eval режим
    model.eval()

    all_predictions = []
    all_labels = []
    for input_ids, labels in dataloader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        preds = model(input_ids)
        
        all_predictions.extend((preds > 0.5).int().cpu())
        all_labels.extend(labels.cpu())

    # усредняем в самом конце, чтобы не зависеть от размера батча
    f1 = compute_f1(torch.stack(all_predictions), torch.stack(all_labels))
    return f1

Как всегда, начать решение задачи стоит с обучения базовой модели, результат которой мы будем улучшать.

__Задание 5.__ Обучите написанную выше модель RNN с помощью реализованных вами функций. Не обязательно обучать модель до полной сходимости, достаточно будет получить F1 больше 0.33. Так как модель очень простая, мы советуем выбирать скорость обучения побольше.

In [21]:
import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

n_classes = 29 # кол-во столбцов - кол-во классов
pad_token_id = dictionary.token2id['PAD']

model = RNN(vocab_size=len(dictionary), n_classes=n_classes, pad_token_id=pad_token_id).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr= 1e-2)#1e2

n_epochs = 20
for epoch in tqdm.tqdm(range(n_epochs)):
    train(model, train_loader, optimizer, device=device)
    f1 = evaluate(model, test_loader, device=device)
    print('##################')
    print('epoch', epoch, 'test f1', f1)
    print('##################')

  0%|          | 0/20 [00:00<?, ?it/s]<ipython-input-13-e455392a376b>:15: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  torch.tensor(labels).float()
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/lo

##################
epoch 0 test f1 0.004068762079137422
##################


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics

##################
epoch 1 test f1 0.008105646007984713
##################


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics

...

##################
epoch 17 test f1 0.1293764883314356
##################


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics

##################
epoch 18 test f1 0.12739693287479464
##################


/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics/_classification.py:1609: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, "true nor predicted", "F-score is", len(true_sum))
/usr/local/lib/python3.10/dist-packages/sklearn/metrics

##################
epoch 19 test f1 0.12521148097566118
##################


In [22]:
torch.save(model.state_dict(), "model.pt")

### GRU

Теперь перейдем к более интересным рекурренным моделям. В практической части урока мы разбирали LSTM и увидели, что она значительно обходит RNN. Вам предлагается предлагается написать и обучить другую популярную рекуррентную модель – GRU. Она объединяет в себе легковестность RNN и идею модуля памяти LSTM и выглядит следующим образом.

<img src="https://i.ibb.co/3FN81P9/gru.png" alt="drawing" width="400"/>

Параметры блока GPU обновляются вот так:
\begin{align}
&z_t =\sigma(W_z x_{t-1} + U_z h_{t-1} + b_z)\\
&r_t =\sigma(W_r x_{t-1} + U_r h_{t-1} + b_r)\\
&\tilde{h}_t = \tanh(W_h x_{t-1} + U_h(r_t \odot h_{t-1}) + b_h)\\
&h_t = (1-z_t) \odot h_{t-1}+ z_t \odot \tilde{h}_t
\end{align}

__Задание 6.__ Реализуйте стандартную GRU, обучите ее с такими же размерами слоев, что и у RNN, и сравните качество. Если вы все сделали правильно, у вас должно получиться F1 больше 0.36. Использовать `nn.GRU` запрещается. Не забудьте про sigmoid на выходе модели.

In [31]:
class GRU(nn.Module):
    def __init__(self, vocab_size, n_classes, pad_token_id, input_size=256, hidden_size=256,
                 num_layers=1, bidirectional=False):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, input_size)

        self.hidden_size = hidden_size

        self.zt = nn.Linear(input_size + hidden_size, hidden_size)
        self.rt = nn.Linear(input_size + hidden_size, hidden_size)
        self.ht = nn.Linear(input_size + hidden_size, hidden_size)

        self.out_fc = nn.Linear(hidden_size, n_classes)

    def forward(self, input_ids, init_h=None):
        bs, seq_len = input_ids.shape
        
        x = self.embedding(input_ids)

        if init_h is None:
            h_t = torch.zeros(bs, self.hidden_size, device=x.device)
        else:
            h_t = init_h
        
        hidden_states = []
        for t in range(seq_len):
            x_t = x[:, t, :]
            # batch the computations into a single matrix multiplication
            cat_state = torch.cat((x_t, h_t), dim=1)
            
            zt_pred = torch.sigmoid(self.zt(cat_state))
            rt_pred = torch.sigmoid(self.rt(cat_state))

            cat_state_2 = torch.cat((x_t, rt_pred * h_t), dim=1)
            ht_t_pred = torch.tanh(self.ht(cat_state_2))
            
            h_t = (1 - zt_pred) * h_t + zt_pred * ht_t_pred
            
            hidden_states.append(h_t.unsqueeze(1))

        hidden_states = torch.cat(hidden_states, dim=1)
        out_hidden_states = self.out_fc(hidden_states)
        
        return out_hidden_states

In [32]:
import tqdm
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

n_classes = 29 # кол-во столбцов - кол-во классов
pad_token_id = dictionary.token2id['PAD']

model = GRU(vocab_size=len(dictionary), n_classes=n_classes, pad_token_id=pad_token_id).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr= 1e-2)#1e2

n_epochs = 20
for epoch in tqdm.tqdm(range(n_epochs)):
    train(model, train_loader, optimizer, device=device)
    f1 = evaluate(model, test_loader, device=device)
    print('##################')
    print('epoch', epoch, 'test f1', f1)
    print('##################')


  0%|          | 0/20 [00:00<?, ?it/s]


ValueError: Using a target size (torch.Size([64, 29])) that is different to the input size (torch.Size([64, 713, 29])) is deprecated. Please ensure they have the same size.

__Задание 7.__ В этом задании у вас есть две опции на выбор: добавить __двунаправленность__ для GRU _или_ добавить __многослойность__. Можно сделать и то, и другое, но дополнительных баллов за это мы не дадим, только бесконечный респект. Обе модификации реализуются довольно просто и дают примерно одинаковый прирост в качестве, поэтому мы не будем вдаваться в подробности. У вас должно получиться качество на тестовой выборке не меньше 0.37.

В грейдере мы будем создавать вашу модель следующим образом:

```
model_path = "это мы тут вашу модель положили где-то.pt"
model = GRU()
model.load_state_dict(torch.load(model_path, map_location=device))
```
Для этого вам нужно задать значения по умолчанию для всех параметров вашей модели. Не забудьте про сигмоиду в конце модели!

In [ ]:
# выберите одно из bidirectional=True и num_layers=2
model = GRU(...).to(device)
optimizer = ...

## Резюме

Если вы добрались досюда, то вы успешно справились со всеми заданиями. Поздравляем!

Вы должны были заметить, что рекуррентные модели, написанные на python, ужасно долго учатся. Все дело в цикле, который их сильно тормозит. Если в будущем вы будете использовать рекуррентные модели, то мы настоятельно рекомендуем брать реализации из pytorch. Они написаны на C и работают намного быстрее.